In [ ]:
from google.colab import drive

In [ ]:
drive.mount('/content/drive')

In [ ]:
!pip install timm torch torchvision --quiet

In [ ]:
!pip install onnx ml_dtypes --upgrade

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
import timm
from tqdm import tqdm
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

In [ ]:
data_dir = "/content/drive/MyDrive/work/Indian_bovine_breeds"

In [ ]:
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(0.2,0.2,0.2),
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

val_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

In [ ]:
full_dataset = datasets.ImageFolder(data_dir, transform=train_transforms)
num_classes = len(full_dataset.classes)
print("Classes:", full_dataset.classes)

In [ ]:
train_size = int(0.7 * len(full_dataset))
val_size = int(0.2 * len(full_dataset))
test_size = len(full_dataset) - train_size - val_size
train_dataset, val_dataset, test_dataset = random_split(full_dataset, [train_size, val_size, test_size])

print(f"Number of images in training set: {len(train_dataset)}")
print(f"Number of images in validation set: {len(val_dataset)}")
print(f"Number of images in test set: {len(test_dataset)}")

In [ ]:
def evaluate_model(model, test_loader, criterion, name):
    model.eval()
    test_loss, correct, total = 0, 0, 0
    with torch.no_grad():
        for images, labels in tqdm(test_loader, desc=f"{name} [Test]"):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            test_loss += loss.item()
            _, preds = outputs.max(1)
            correct += preds.eq(labels).sum().item()
            total += labels.size(0)
    test_acc = 100.*correct/total
    print(f"[{name}] Test Acc: {test_acc:.2f}% | Test Loss: {test_loss/len(test_loader):.4f}")
    return test_acc

# Example usage (assuming 'best_model' and 'model' from previous steps are available)
# evaluate_model(model, test_loader, criterion, best_model)

In [ ]:
train_dataset.dataset.transform = train_transforms
val_dataset.dataset.transform   = val_transforms

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
val_loader   = DataLoader(val_dataset,   batch_size=32, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_dataset,  batch_size=32, shuffle=False, num_workers=2)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
def train_model_with_history(model, optimizer, criterion, epochs=15, name="model", patience=5):
    history = {"train_acc": [], "val_acc": [], "train_loss": [], "val_loss": []}
    best_acc = 0
    epochs_no_improve = 0

    for epoch in range(epochs):
        # ---------- Training ----------
        model.train()
        train_loss, correct, total = 0, 0, 0
        for images, labels in tqdm(train_loader, desc=f"{name} Epoch {epoch+1}/{epochs} [Train]"):
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            _, preds = outputs.max(1)
            correct += preds.eq(labels).sum().item()
            total += labels.size(0)
        train_acc = 100.*correct/total

        # ---------- Validation ----------
        model.eval()
        val_loss, correct, total = 0, 0, 0
        with torch.no_grad():
            for images, labels in tqdm(val_loader, desc=f"{name} Epoch {epoch+1}/{epochs} [Val]"):
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)
                val_loss += loss.item()
                _, preds = outputs.max(1)
                correct += preds.eq(labels).sum().item()
                total += labels.size(0)
        val_acc = 100.*correct/total

        # ---------- Save history ----------
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)
        history["train_loss"].append(train_loss/len(train_loader))
        history["val_loss"].append(val_loss/len(val_loader))

        print(f"[{name}] Epoch {epoch+1}/{epochs} | "
              f"Train Acc: {train_acc:.2f}% | Val Acc: {val_acc:.2f}% | "
              f"Train Loss: {train_loss/len(train_loader):.4f} | Val Loss: {val_loss/len(val_loader):.4f}")

        # ---------- Check improvement and early stopping ----------
        if val_acc > best_acc:
            best_acc = val_acc
            epochs_no_improve = 0
            torch.save(model.state_dict(), f"/content/drive/MyDrive/work/cattle_breed/best_{name}.pth")
        else:
            epochs_no_improve += 1
            if epochs_no_improve == patience:
                print(f"[{name}] Early stopping after {patience} epochs without improvement.")
                break

    return history, best_acc

In [ ]:
def plot_history(history, name):
    epochs_range = range(1, len(history["train_acc"]) + 1)
    plt.figure(figsize=(12,5))
    plt.subplot(1,2,1)
    plt.plot(epochs_range, history["train_acc"], label="Train Acc")
    plt.plot(epochs_range, history["val_acc"], label="Val Acc")
    plt.xlabel("Epochs"); plt.ylabel("Accuracy (%)"); plt.title(f"{name} Accuracy"); plt.legend()

    plt.subplot(1,2,2)
    plt.plot(epochs_range, history["train_loss"], label="Train Loss")
    plt.plot(epochs_range, history["val_loss"], label="Val Loss")
    plt.xlabel("Epochs"); plt.ylabel("Loss"); plt.title(f"{name} Loss"); plt.legend()
    plt.show()

In [ ]:
results = {}
histories = {}
criterion = nn.CrossEntropyLoss()

In [ ]:
# ConvNeXt Large with Advanced Settings
convnext_large = timm.create_model("convnext_large", pretrained=True, num_classes=num_classes).to(device)

# Advanced optimizer with learning rate scheduler
optimizer_large = optim.AdamW(convnext_large.parameters(), lr=2e-5, weight_decay=0.05)
scheduler_large = optim.lr_scheduler.OneCycleLR(
    optimizer_large,
    max_lr=1e-4,
    epochs=20,
    steps_per_epoch=len(train_loader),
    pct_start=0.2
)

history_large, acc_large = train_model_with_history(
    convnext_large,
    optimizer_large,
    criterion,
    epochs=20,
    name="convnext_large",
    patience=7 # Added patience parameter
)

plot_history(history_large, "ConvNeXt_Large")
results["ConvNeXt_Large"] = acc_large
histories["ConvNeXt_Large"] = history_large

In [ ]:
print("\n==== Final Validation Accuracies ====")
for model_name, acc in results.items():
    print(f"{model_name}: {acc:.2f}%")

best_model = max(results, key=results.get)
print(f"\n✅ Best Model: {best_model} with {results[best_model]:.2f}% validation accuracy")

In [ ]:

def compare_models(histories, metric="acc"):
    plt.figure(figsize=(12,6))

    for name, history in histories.items():
        if metric == "acc":
            plt.plot(history["train_acc"], label=f"{name} Train")
            plt.plot(history["val_acc"], linestyle="--", label=f"{name} Val")
            plt.ylabel("Accuracy (%)")
            plt.title("Train vs Val Accuracy Comparison")
        elif metric == "loss":
            plt.plot(history["train_loss"], label=f"{name} Train")
            plt.plot(history["val_loss"], linestyle="--", label=f"{name} Val")
            plt.ylabel("Loss")
            plt.title("Train vs Val Loss Comparison")

    plt.xlabel("Epochs")
    plt.legend()
    plt.show()

In [ ]:
# Accuracy comparison
compare_models(histories, metric="acc")

In [ ]:
# Loss comparison
compare_models(histories, metric="loss")

In [ ]:
plt.figure(figsize=(8,6))
plt.bar(results.keys(), results.values(), color=["skyblue","lightgreen","salmon","orange"])
plt.ylabel("Validation Accuracy (%)")
plt.title("Final Validation Accuracy Comparison")
plt.xticks(rotation=20)
plt.show()

# =============================
# STEP 8: Inference with Unknown Detection
# =============================

In [ ]:
def predict_image(image_path, model, class_names, threshold=0.7):
    """
    Predicts the class of an image, or flags as unknown if confidence < threshold.
    """
    model.eval()
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406],
                             [0.229, 0.224, 0.225])
    ])

    img = Image.open(image_path).convert("RGB")
    img_tensor = transform(img).unsqueeze(0).to(device)

    with torch.no_grad():
        outputs = model(img_tensor)
        probs = torch.nn.functional.softmax(outputs, dim=1)
        max_prob, pred_class = torch.max(probs, 1)

    max_prob = max_prob.item()
    pred_class = pred_class.item()

    if max_prob < threshold:
        return "Unknown Image — Not a recognized cattle/buffalo breed"
    else:
        return f"Predicted: {class_names[pred_class]} (Confidence: {max_prob:.2f})"

# Load the best model (update with correct model name if needed)

In [ ]:
model = timm.create_model(best_model.lower(), pretrained=False, num_classes=num_classes).to(device)
model.load_state_dict(torch.load(f"/content/drive/MyDrive/work/cattle_breed/best_{best_model.lower()}.pth"))
model.eval()

# Predict on a new image

In [ ]:
from google.colab import files

uploaded = files.upload()

for fn in uploaded.keys():
  print('User uploaded file "{name}" with length {length} bytes'.format(
      name=fn, length=len(uploaded[fn])))
  # Use the first uploaded file for prediction
  test_img = fn

In [ ]:
result = predict_image(test_img, model, full_dataset.classes, threshold=0.7)
print(result)

In [ ]:
import json

# Export Best Model for Deployment

In [ ]:
best_model_name = best_model  # update if best was different
weights_path = f"/content/drive/MyDrive/work/cattle_breed/best_{best_model_name.lower()}.pth"

In [ ]:
example_input = torch.randn(1, 3, 224, 224).to(device)
traced_model = torch.jit.trace(model, example_input)
torchscript_path = f"/content/drive/MyDrive/work/cattle_breed/best_{best_model_name.lower()}_scripted.pt"
traced_model.save(torchscript_path)

In [ ]:
model = timm.create_model(best_model_name.lower(), pretrained=False, num_classes=num_classes).to(device)
model.load_state_dict(torch.load(weights_path))
model.eval()

In [ ]:
print(f"TorchScript model saved to: {torchscript_path}")

In [ ]:
metadata = {
    "model_name": "swin_small_patch4_window7_224",
    "num_classes": num_classes,
    "class_labels": full_dataset.classes,
    "input_size": [3, 224, 224],
    "normalization": {
        "mean": [0.485, 0.456, 0.406],
        "std": [0.229, 0.224, 0.225]
    }
}

In [ ]:
json_path = f"/content/drive/MyDrive/work/cattle_breed/{best_model_name}_metadata.json"
with open(json_path, "w") as f:
    json.dump(metadata, f, indent=4)

In [ ]:
print(f"Metadata JSON saved to: {json_path}")

In [ ]:
onnx_path = f"/content/drive/MyDrive/work/cattle_breed/{best_model_name}.onnx"
torch.onnx.export(
    model,
    example_input,
    onnx_path,
    input_names=["input"],
    output_names=["output"],
    dynamic_axes={"input": {0: "batch"}, "output": {0: "batch"}},
    opset_version=13
)
print(f"ONNX model saved to: {onnx_path}")

Now that the model is loaded correctly, you can proceed with testing it on a new image and exporting it for deployment.